In [ ]:
# ============================================================
# MINI-PROJECT 2: SOLAR MICRO-GRID DISPATCH PLANNER
# Rural Health Centre - Kasese
# ============================================================

import math
import timeit
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.linalg import solve
from scipy.optimize import nnls
from statistics import mean, variance, stdev


# ============================================================
# 1. MICRO-GRID CLASS
# ============================================================

class MicroGrid:
    """
    Solar + battery micro-grid model.

    Equations:

        3x + 2y = D1
        4x +  y = D2

    x = solar energy
    y = battery energy
    """

    def __init__(self):
        self.A = np.array([
            [3.0, 2.0],
            [4.0, 1.0]
        ])

    def check_system(self):
        """
        Check determinant and condition number.
        """

        determinant = np.linalg.det(self.A)
        condition_number = np.linalg.cond(self.A)

        print("=" * 70)
        print("SYSTEM WELL-POSEDNESS")
        print("=" * 70)

        print("\nCoefficient matrix A:")
        print(self.A)

        print(f"\nDeterminant: {determinant:.6f}")
        print(f"Condition number: {condition_number:.6f}")

        if abs(determinant) < 1e-12:
            print("\nWARNING: The matrix is singular.")
            print("The system does not have a unique solution.")
        else:
            print("\nThe determinant is non-zero.")
            print("Therefore, the system has a unique solution.")

        if condition_number < 10:
            print("The system is well-conditioned.")
        elif condition_number < 100:
            print("The system has moderate sensitivity.")
        else:
            print("WARNING: The system may be ill-conditioned.")

        return determinant, condition_number

    def solve_day(self, d1, d2):
        """
        Solve one day's dispatch using scipy.linalg.solve.
        """

        b = np.array([d1, d2], dtype=float)

        x, y = solve(self.A, b)

        return x, y

    def solve_days_loop(self, demands):
        """
        Solve multiple days using a Python loop.
        """

        results = []

        for d1, d2 in demands:
            x, y = self.solve_day(d1, d2)
            results.append([x, y])

        return np.array(results)

    def solve_days_vectorized(self, demands):
        """
        Solve all days at once.

        scipy.linalg.solve accepts a matrix B whose columns
        contain multiple right-hand sides.
        """

        d1 = demands[:, 0]
        d2 = demands[:, 1]

        B = np.vstack([d1, d2])

        solution = solve(self.A, B)

        return solution.T


# ============================================================
# 2. CREATE MICRO-GRID
# ============================================================

grid = MicroGrid()

grid.check_system()


# ============================================================
# 3. INTERACTIVE INPUT
# ============================================================

def get_positive_number(prompt):
    """
    Robustly request a positive numeric value.
    """

    while True:

        value = input(prompt).strip()

        if value == "":
            print("Error: value cannot be empty.")
            continue

        try:
            number = float(value)

        except ValueError:
            print("Error: please enter a numeric value.")
            continue

        if number < 0:
            print("Error: negative values are not allowed.")
            continue

        return number


def interactive_day():

    print("\n" + "=" * 70)
    print("INTERACTIVE DAILY DEMAND INPUT")
    print("=" * 70)

    d1 = get_positive_number(
        "Enter D1 daytime load (kWh): "
    )

    d2 = get_positive_number(
        "Enter D2 critical-equipment load (kWh): "
    )

    x, y = grid.solve_day(d1, d2)

    print("\nMathematical solution:")
    print(f"Solar:   {x:.2f} kWh")
    print(f"Battery: {y:.2f} kWh")

    if x < 0 or y < 0:
        print("\nWARNING: This solution is physically infeasible.")
        print("At least one energy source is negative.")
    else:
        print("\nThe solution is physically feasible.")


# Uncomment the following line if you want interactive input.
# interactive_day()


# ============================================================
# 4. GENERATE 30 DAYS OF DEMAND
# ============================================================

def generate_demand_csv(
    filename="kasese_microgrid_demand.csv",
    days=30,
    seed=42
):
    """
    Generate realistic synthetic demand data.

    A weekly pattern is included:
    weekdays have slightly higher demand than weekends.

    Random noise is added using a seeded generator so
    results are reproducible.
    """

    rng = np.random.default_rng(seed)

    day_numbers = np.arange(1, days + 1)

    # Day of week:
    # 0 = Monday ... 6 = Sunday
    day_of_week = np.arange(days) % 7

    # Weekly demand pattern
    weekly_factor = np.where(
        day_of_week < 5,
        1.00,     # weekdays
        0.88      # weekends
    )

    # Base demand
    base_d1 = 100
    base_d2 = 120

    # Random noise
    noise_d1 = rng.normal(
        loc=0,
        scale=5,
        size=days
    )

    noise_d2 = rng.normal(
        loc=0,
        scale=6,
        size=days
    )

    d1 = (
        base_d1
        * weekly_factor
        + noise_d1
    )

    d2 = (
        base_d2
        * weekly_factor
        + noise_d2
    )

    # Make sure demands cannot be negative
    d1 = np.maximum(d1, 1)
    d2 = np.maximum(d2, 1)

    df = pd.DataFrame({
        "Day": day_numbers,
        "D1_kWh": d1,
        "D2_kWh": d2
    })

    df.to_csv(
        filename,
        index=False
    )

    print("\n" + "=" * 70)
    print("30-DAY DEMAND CSV CREATED")
    print("=" * 70)

    print(f"\nFile: {filename}")
    print("\nFirst five days:")
    print(df.head())

    return df


demand_df = generate_demand_csv()


# ============================================================
# 5. LOAD CSV
# ============================================================

def load_demand_csv(filename):
    """
    Load and validate the demand CSV.
    """

    df = pd.read_csv(filename)

    required_columns = [
        "Day",
        "D1_kWh",
        "D2_kWh"
    ]

    for column in required_columns:
        if column not in df.columns:
            raise ValueError(
                f"Missing required column: {column}"
            )

    if len(df) != 30:
        print(
            f"WARNING: Expected 30 days, "
            f"but found {len(df)}."
        )

    if df[["D1_kWh", "D2_kWh"]].isnull().any().any():
        raise ValueError(
            "Demand data contains missing values."
        )

    if (df[["D1_kWh", "D2_kWh"]] < 0).any().any():
        raise ValueError(
            "Demand values cannot be negative."
        )

    return df


demand_df = load_demand_csv(
    "kasese_microgrid_demand.csv"
)

print("\nLoaded demand data:")
print(demand_df)


# ============================================================
# 6. SOLVE 30 DAYS USING PYTHON LOOP
# ============================================================

demands = demand_df[
    ["D1_kWh", "D2_kWh"]
].to_numpy()

loop_results = grid.solve_days_loop(
    demands
)


# ============================================================
# 7. SOLVE 30 DAYS USING VECTORIZED SOLVE
# ============================================================

vector_results = grid.solve_days_vectorized(
    demands
)


# ============================================================
# 8. COMPARE LOOP AND VECTORIZED RESULTS
# ============================================================

difference = np.max(
    np.abs(loop_results - vector_results)
)

print("\n" + "=" * 70)
print("LOOP VS VECTORIZED SOLUTION")
print("=" * 70)

print(
    f"\nMaximum difference: {difference:.12f}"
)

if difference < 1e-10:
    print("Both methods give the same numerical results.")


# ============================================================
# 9. TIME BOTH APPROACHES
# ============================================================

def run_loop():
    grid.solve_days_loop(demands)


def run_vectorized():
    grid.solve_days_vectorized(demands)


loop_time = timeit.timeit(
    run_loop,
    number=1000
)

vector_time = timeit.timeit(
    run_vectorized,
    number=1000
)

print("\n" + "=" * 70)
print("PERFORMANCE COMPARISON")
print("=" * 70)

print(
    f"\nLoop time for 1000 runs:       "
    f"{loop_time:.6f} seconds"
)

print(
    f"Vectorized time for 1000 runs: "
    f"{vector_time:.6f} seconds"
)

if vector_time > 0:
    speedup = loop_time / vector_time

    print(
        f"Speed-up from vectorisation: "
        f"{speedup:.2f}x"
    )

print("""
Comment:
The Python loop repeatedly calls the solver for each day.
The vectorised approach sends all 30 right-hand sides to the
linear algebra routine in one operation.

For only 30 days, the difference may be small because the
dataset is small. Vectorisation becomes more valuable for
larger datasets because it reduces Python-level looping and
allows optimized numerical libraries to do more of the work.
""")


# ============================================================
# 10. PHYSICAL FEASIBILITY
# ============================================================

results_df = demand_df.copy()

results_df["Solar_kWh"] = loop_results[:, 0]
results_df["Battery_kWh"] = loop_results[:, 1]

results_df["Infeasible"] = (
    (results_df["Solar_kWh"] < 0)
    |
    (results_df["Battery_kWh"] < 0)
)


print("\n" + "=" * 70)
print("PHYSICAL FEASIBILITY CHECK")
print("=" * 70)

infeasible_days = results_df[
    results_df["Infeasible"]
]

if len(infeasible_days) == 0:

    print("\nNo physically infeasible days were detected.")

else:

    print(
        f"\n{len(infeasible_days)} "
        "physically infeasible day(s) detected."
    )

    print(
        infeasible_days[
            [
                "Day",
                "D1_kWh",
                "D2_kWh",
                "Solar_kWh",
                "Battery_kWh"
            ]
        ]
    )


# ============================================================
# 11. NON-NEGATIVE LEAST-SQUARES HANDLING
# ============================================================

def solve_nonnegative(d1, d2):

    b = np.array([
        d1,
        d2
    ])

    solution, residual_norm = nnls(
        grid.A,
        b
    )

    return solution[0], solution[1], residual_norm


nnls_results = []

for _, row in demand_df.iterrows():

    x, y, residual = solve_nonnegative(
        row["D1_kWh"],
        row["D2_kWh"]
    )

    nnls_results.append([
        x,
        y,
        residual
    ])


nnls_results = np.array(
    nnls_results
)

results_df["NNLS_Solar_kWh"] = (
    nnls_results[:, 0]
)

results_df["NNLS_Battery_kWh"] = (
    nnls_results[:, 1]
)

results_df["NNLS_Residual"] = (
    nnls_results[:, 2]
)


print("\n" + "=" * 70)
print("NON-NEGATIVE LEAST-SQUARES RESULTS")
print("=" * 70)

print("""
NNLS is used when the mathematical solution gives negative
energy values.

Unlike ordinary linear solving, NNLS forces:

    x >= 0
    y >= 0

The trade-off is that when the original equations cannot be
satisfied with non-negative values, NNLS minimizes the
least-squares error instead.
""")


# ============================================================
# 12. USE NNLS AS FINAL DISPATCH
# ============================================================

results_df["Solar_Final_kWh"] = results_df[
    "NNLS_Solar_kWh"
]

results_df["Battery_Final_kWh"] = results_df[
    "NNLS_Battery_kWh"
]


# ============================================================
# 13. STATISTICS
# ============================================================

solar = results_df[
    "Solar_Final_kWh"
].to_numpy()

battery = results_df[
    "Battery_Final_kWh"
].to_numpy()


print("\n" + "=" * 70)
print("ENERGY USAGE STATISTICS")
print("=" * 70)


def print_statistics(name, data):

    print(f"\n{name}")

    print(
        f"Mean:              "
        f"{mean(data):.2f} kWh"
    )

    print(
        f"Variance:          "
        f"{variance(data):.2f}"
    )

    print(
        f"Standard deviation:"
        f" {stdev(data):.2f} kWh"
    )


print_statistics(
    "Solar",
    solar
)

print_statistics(
    "Battery",
    battery
)


solar_std = stdev(solar)
battery_std = stdev(battery)


print("\nVolatility comparison:")

if solar_std > battery_std:

    print(
        "Solar usage is more volatile "
        "because it has the larger standard deviation."
    )

elif battery_std > solar_std:

    print(
        "Battery usage is more volatile "
        "because it has the larger standard deviation."
    )

else:

    print(
        "Solar and battery usage have equal volatility."
    )


# ============================================================
# 14. COST MODEL
# ============================================================

SOLAR_COST = 150
BATTERY_COST = 450


results_df["Solar_Cost_UGX"] = (
    results_df["Solar_Final_kWh"]
    * SOLAR_COST
)

results_df["Battery_Cost_UGX"] = (
    results_df["Battery_Final_kWh"]
    * BATTERY_COST
)

results_df["Daily_Cost_UGX"] = (
    results_df["Solar_Cost_UGX"]
    + results_df["Battery_Cost_UGX"]
)


monthly_cost = results_df[
    "Daily_Cost_UGX"
].sum()


print("\n" + "=" * 70)
print("COST MODEL")
print("=" * 70)

print(
    f"\nSolar cost: "
    f"UGX {SOLAR_COST:,}/kWh"
)

print(
    f"Battery cost: "
    f"UGX {BATTERY_COST:,}/kWh"
)

print(
    f"\n30-day energy cost: "
    f"UGX {monthly_cost:,.2f}"
)

print(
    f"Average daily cost: "
    f"UGX {results_df['Daily_Cost_UGX'].mean():,.2f}"
)


# ============================================================
# 15. DAILY RESULTS TABLE
# ============================================================

print("\n" + "=" * 70)
print("DAILY DISPATCH RESULTS")
print("=" * 70)

display_columns = [
    "Day",
    "D1_kWh",
    "D2_kWh",
    "Solar_Final_kWh",
    "Battery_Final_kWh",
    "Daily_Cost_UGX"
]

print(
    results_df[
        display_columns
    ].round(2).to_string(index=False)
)


# ============================================================
# 16. STACKED BAR CHART + DAILY COST
# ============================================================

fig, ax1 = plt.subplots(
    figsize=(15, 7)
)

days = results_df["Day"]

ax1.bar(
    days,
    results_df["Solar_Final_kWh"],
    label="Solar",
    color="gold"
)

ax1.bar(
    days,
    results_df["Battery_Final_kWh"],
    bottom=results_df["Solar_Final_kWh"],
    label="Battery",
    color="steelblue"
)

ax1.set_xlabel("Day")
ax1.set_ylabel(
    "Energy usage (kWh)"
)

ax1.set_title(
    "Kasese Micro-Grid Daily Energy Dispatch and Cost"
)

ax1.legend(
    loc="upper left"
)

ax1.grid(
    axis="y",
    alpha=0.3
)


# Second axis for cost

ax2 = ax1.twinx()

ax2.plot(
    days,
    results_df["Daily_Cost_UGX"],
    color="red",
    marker="o",
    linewidth=2,
    label="Daily Cost"
)

ax2.set_ylabel(
    "Daily cost (UGX)",
    color="red"
)

ax2.tick_params(
    axis="y",
    labelcolor="red"
)


plt.tight_layout()
plt.show()


# ============================================================
# 17. HYBRID MICRO-GRID EXTENSION
# ============================================================

class HybridMicroGrid(MicroGrid):
    """
    Hybrid system with:

        3x + 2y + z = D1
        4x +  y      = D2
        x  +  y + z  = D3

    x = solar
    y = battery
    z = diesel
    """

    def __init__(self):

        self.A = np.array([
            [3.0, 2.0, 1.0],
            [4.0, 1.0, 0.0],
            [1.0, 1.0, 1.0]
        ])

    def solve_day(self, d1, d2, d3):

        b = np.array([
            d1,
            d2,
            d3
        ])

        x, y, z = solve(
            self.A,
            b
        )

        return x, y, z


# ============================================================
# 18. TEST HYBRID MICRO-GRID
# ============================================================

hybrid = HybridMicroGrid()

print("\n" + "=" * 70)
print("HYBRID MICRO-GRID")
print("=" * 70)

print("\nHybrid coefficient matrix:")
print(hybrid.A)

hybrid_determinant = np.linalg.det(
    hybrid.A
)

hybrid_condition = np.linalg.cond(
    hybrid.A
)

print(
    f"\nDeterminant: "
    f"{hybrid_determinant:.6f}"
)

print(
    f"Condition number: "
    f"{hybrid_condition:.6f}"
)


x, y, z = hybrid.solve_day(
    100,
    120,
    50
)

print("\nExample hybrid solution:")
print(f"Solar:   {x:.2f} kWh")
print(f"Battery: {y:.2f} kWh")
print(f"Diesel:  {z:.2f} kWh")


# ============================================================
# 19. LINEAR DEPENDENCE DEMONSTRATION
# ============================================================

dependent_matrix = np.array([
    [3.0, 2.0, 1.0],
    [4.0, 1.0, 0.0],
    [7.0, 3.0, 1.0]
])

dependent_det = np.linalg.det(
    dependent_matrix
)

dependent_cond = np.linalg.cond(
    dependent_matrix
)

print("\n" + "=" * 70)
print("LINEAR DEPENDENCE TEST")
print("=" * 70)

print("\nDependent matrix:")
print(dependent_matrix)

print(
    f"\nDeterminant: "
    f"{dependent_det:.6f}"
)

print(
    f"Condition number: "
    f"{dependent_cond:.6e}"
)

print("""
The third row is the sum of the first and second rows:

[7, 3, 1] = [3, 2, 1] + [4, 1, 0]

Therefore the equations are linearly dependent.

The determinant is zero (up to floating-point precision),
so the matrix is singular and the system does not have a
unique solution.

This means the new constraint does not provide independent
information.
""")


# ============================================================
# 20. MONTE CARLO SENSITIVITY ANALYSIS
# ============================================================

def monte_carlo_sensitivity(
    d1,
    d2,
    n=1000,
    seed=42
):

    rng = np.random.default_rng(seed)

    # Random percentage changes between -5% and +5%
    changes = rng.uniform(
        -0.05,
        0.05,
        size=(n, 2)
    )

    perturbed_d1 = (
        d1 * (1 + changes[:, 0])
    )

    perturbed_d2 = (
        d2 * (1 + changes[:, 1])
    )

    B = np.vstack([
        perturbed_d1,
        perturbed_d2
    ])

    solutions = solve(
        grid.A,
        B
    )

    solar_values = solutions[0]
    battery_values = solutions[1]

    return (
        perturbed_d1,
        perturbed_d2,
        solar_values,
        battery_values
    )


# Use the first day's demand as the baseline

base_d1 = demand_df.iloc[0]["D1_kWh"]
base_d2 = demand_df.iloc[0]["D2_kWh"]


(
    perturbed_d1,
    perturbed_d2,
    mc_solar,
    mc_battery
) = monte_carlo_sensitivity(
    base_d1,
    base_d2,
    n=1000,
    seed=42
)


# ============================================================
# 21. SENSITIVITY RESULTS
# ============================================================

print("\n" + "=" * 70)
print("MONTE CARLO SENSITIVITY ANALYSIS")
print("=" * 70)

print(
    f"\nBaseline D1: "
    f"{base_d1:.2f} kWh"
)

print(
    f"Baseline D2: "
    f"{base_d2:.2f} kWh"
)


base_x, base_y = grid.solve_day(
    base_d1,
    base_d2
)

print(
    f"\nBaseline solar: "
    f"{base_x:.2f} kWh"
)

print(
    f"Baseline battery: "
    f"{base_y:.2f} kWh"
)


solar_change = (
    mc_solar - base_x
)

battery_change = (
    mc_battery - base_y
)


print("\nSolar sensitivity:")

print(
    f"Mean change: "
    f"{np.mean(solar_change):.4f} kWh"
)

print(
    f"Std change: "
    f"{np.std(solar_change):.4f} kWh"
)

print(
    f"Minimum change: "
    f"{np.min(solar_change):.4f} kWh"
)

print(
    f"Maximum change: "
    f"{np.max(solar_change):.4f} kWh"
)


print("\nBattery sensitivity:")

print(
    f"Mean change: "
    f"{np.mean(battery_change):.4f} kWh"
)

print(
    f"Std change: "
    f"{np.std(battery_change):.4f} kWh"
)

print(
    f"Minimum change: "
    f"{np.min(battery_change):.4f} kWh"
)

print(
    f"Maximum change: "
    f"{np.max(battery_change):.4f} kWh"
)


# ============================================================
# 22. SENSITIVITY PLOTS
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5)
)


axes[0].hist(
    solar_change,
    bins=30,
    color="gold",
    edgecolor="black",
    alpha=0.8
)

axes[0].axvline(
    0,
    color="black",
    linestyle="--"
)

axes[0].set_title(
    "Solar Change from ±5% Demand Perturbations"
)

axes[0].set_xlabel(
    "Change in solar usage (kWh)"
)

axes[0].set_ylabel(
    "Frequency"
)


axes[1].hist(
    battery_change,
    bins=30,
    color="steelblue",
    edgecolor="black",
    alpha=0.8
)

axes[1].axvline(
    0,
    color="black",
    linestyle="--"
)

axes[1].set_title(
    "Battery Change from ±5% Demand Perturbations"
)

axes[1].set_xlabel(
    "Change in battery usage (kWh)"
)

axes[1].set_ylabel(
    "Frequency"
)


plt.tight_layout()
plt.show()


# ============================================================
# 23. FINAL PROJECT SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("PROJECT SUMMARY")
print("=" * 70)

print("""
1. The MicroGrid class represents the two-equation solar/battery
   dispatch system.

2. The determinant checks whether the coefficient matrix is
   singular. A non-zero determinant means a unique solution exists.

3. The condition number measures sensitivity to small changes
   in the input data.

4. Interactive input validates numeric, non-negative values.

5. A seeded random generator creates 30 days of synthetic
   Kasese demand data with a weekly pattern and random noise.

6. The 30 days are solved using both a Python loop and a
   vectorised scipy.linalg.solve call.

7. Physically infeasible negative energy values are detected.

8. Non-negative least squares (NNLS) provides a practical
   alternative that enforces solar >= 0 and battery >= 0.

9. Mean, variance and standard deviation are calculated to
   compare energy-source volatility.

10. Energy costs are calculated using:
        Solar   = UGX 150/kWh
        Battery = UGX 450/kWh

11. A stacked bar chart shows daily solar and battery usage,
    while a second axis shows daily cost.

12. The HybridMicroGrid extension adds a diesel generator and
    a third constraint.

13. If the third equation is linearly dependent on the first
    two, the determinant becomes zero and there is no unique
    solution.

14. Monte Carlo sensitivity analysis perturbs D1 and D2 by
    ±5% using 1,000 simulations and measures the resulting
    changes in solar and battery requirements.
""")

print("\nProject completed successfully.")
